In [1]:
import os
import re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
import loguru as logger

from settings.settings_analyze_efizz import Settings_ae
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.rayleigh.load_rayleigh import collect_all_rayleigh_paths, load_all_rayleigh_data
from behave_analysis.utils.rayleigh.manipulate_rayleigh_df import extract_compartment_values, extract_firing_rates
from behave_analysis.utils.creating_directories import make_directory
from settings.settings_analyze_efizz import Settings_ae as Settings
from behave_analysis.analyze.TunED.model import TunEdModel

In [2]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

In [5]:
experiments_objects = [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

tinny_barrier = [JAL8_tiny_3may, JAL8_21may, JAL7_30apr]

# Mice groups based on session names
mice_groups = {
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept', 'JAL005_8thSept', 'JAL005_21stSept'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}


session_names = ["JAL6_flip7_1apr", "JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept", "JAL005_8thSept", "JAL005_21stSept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

In [4]:
def regex(angle):
    "Remove unwanted characters from angle file string"
    pattern = r'^(.*?)(?=_Rayleigh)'
    match = re.search(pattern, angle)
    assert match, f"Could not find match for {angle}"
    return match.group(0)

def nest_dic():
    """A function to create arbitrarily nested dictionaries"""
    return defaultdict(nest_dic)

In [6]:
# Init Params
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
total_cells = 0
total_sessions = 0
rayleigh_threshold = 0.15
fr_threshold = 5 # Hz
dir = make_directory(r"Z:\Jasmine_Laurence\rayleigh_analysis")

In [7]:
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

In [ ]:
# TODO firing rate threshold not implemented

dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}

cell_count = 0
# For each experiment object 
for i, session in enumerate(experiments_objects):
    
    # load the session
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # Count the number of sessions and cells
    total_sessions += 1
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0
            for angle in angle_keys:
                
                #Just the threat zone
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for each compartment
                if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                    rayleigh = output[cell][1]
                    max_angle_str = regex(angle)                

                # # Whole arena rayleigh
                # output = condition_data[condition][angle]["arena_rayleigh"] # Whole arena rayleigh
                # if np.logical_and(output[cell] > rayleigh, output[cell] > rayleigh_threshold):
                #     rayleigh = output[cell]
                #     max_angle_str = regex(angle)
                    
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                dict[i][cell][condition] = max_angle_str
                                
            except:
                print(f"Cell {cell} in session {i} did not meet threshold")

In [ ]:
# Counts across sessions
shelter = []
barrier_pre_flip = []
barrier_post_flip = []
for session in dict:
    for cell in dict[session]:
        x = dict[session][cell]["shelter_only"]
        shelter.append(x)
        y = dict[session][cell]["barrier_pre_flip"]
        barrier_pre_flip.append(y)
        z = dict[session][cell]["barrier_post_flip"]
        barrier_post_flip.append(z)
shelter_counts = Counter(shelter)
barrier_pre_flip_counts = Counter(barrier_pre_flip)
barrier_post_flip_counts = Counter(barrier_post_flip)
print("Across session counts")
print(shelter_counts)

In [ ]:
# Counts within sessions
within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

for session in dict:
    for cell in dict[session]:
        x = dict[session][cell]["shelter_only"]
        y = dict[session][cell]["barrier_pre_flip"]
        z = dict[session][cell]["barrier_post_flip"]
        within_session_counts_shelter[session][x] += 1
        within_session_counts_bar_pre_flip[session][y] += 1
        within_session_counts_bar_post_flip[session][z] += 1

In [ ]:
xcoords = [0, 1, 3, 4]
plt.bar(xcoords, 
        [barrier_pre_flip_counts['h_preflipbar_a'] / cell_count, 
         barrier_pre_flip_counts['h_postflipbar_a'] / cell_count, 
         barrier_post_flip_counts['h_preflipbar_a'] / cell_count, 
         barrier_post_flip_counts['h_postflipbar_a'] / cell_count],
        color= 'darkorchid',
        alpha = 0.9
        )
plt.xticks(xcoords, 
           ['Open Edge | Pre flip condition', 
            'Closed Edge | Pre flip condition', 
            'Closed Edge | Post flip condition', 
            'Open Edge | Post flip condition'],
           rotation=10)
plt.ylabel('Fraction of cells', fontsize=16)

# Counts within sessions
within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

mouse_dict = defaultdict(list)

for session, session_name in zip(dict, session_names):
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
    
    for cell in dict[session]:
        x = dict[session][cell]["shelter_only"]
        y = dict[session][cell]["barrier_pre_flip"]
        z = dict[session][cell]["barrier_post_flip"]
        within_session_counts_shelter[session][x] += 1
        within_session_counts_bar_pre_flip[session][y] += 1
        within_session_counts_bar_post_flip[session][z] += 1
    
    # Scatter plot points
    y_scatter_values = [
        within_session_counts_bar_pre_flip[session]['h_preflipbar_a'] / sum(within_session_counts_bar_pre_flip[session].values()),
        within_session_counts_bar_pre_flip[session]['h_postflipbar_a'] / sum(within_session_counts_bar_pre_flip[session].values()),
        within_session_counts_bar_post_flip[session]['h_preflipbar_a'] / sum(within_session_counts_bar_post_flip[session].values()),
        within_session_counts_bar_post_flip[session]['h_postflipbar_a'] / sum(within_session_counts_bar_post_flip[session].values())
    ]
        
    plt.scatter(xcoords, y_scatter_values, color='grey', s=100, alpha=1, marker="x")
    
    # Draw lines between scatter points for each session
    plt.plot(xcoords[:2], y_scatter_values[:2], color='grey', alpha=0.5)  # Connect points at 0 and 1
    plt.plot(xcoords[2:], y_scatter_values[2:], color='grey', alpha=0.5)  # Connect points at 3 and 4
    
    mouse_dict[mouse].append(y_scatter_values)

# Apply average mouse information
for mouse in mouse_dict:
    yvals = np.mean(np.array(mouse_dict[mouse]), axis=0)
    plt.plot(xcoords[:2], yvals[:2], linestyle='-', color="black", linewidth=2.5, label=f'{mouse} Average')
    plt.plot(xcoords[2:], yvals[2:], linestyle='-', color="black", linewidth=2.5)